# Clase 113 — Optimizadores: Momentum, Nesterov, AdaGrad, RMSProp, Adam, AdamW

Evolución de los optimizadores y cuándo usar cada uno. **Adam/AdamW** para casi todo, **SGD+Momentum** para visión clásica, **Lion** (2023) para LLMs (menos memoria, LR más bajo).

Requiere: `tensorflow` / `keras` (≥ 3.0, incluye `Lion`), `numpy`.

## 1. Construir la familia completa

De SGD a AdamW y Lion. Cada uno resuelve una limitación del anterior.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

optimizadores = {
    "SGD":      keras.optimizers.SGD(learning_rate=0.01),
    "Momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "Nesterov": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    "AdaGrad":  keras.optimizers.Adagrad(learning_rate=0.01),
    "RMSProp":  keras.optimizers.RMSprop(learning_rate=1e-3, rho=0.9),
    "Adam":     keras.optimizers.Adam(learning_rate=1e-3),
    "AdamW":    keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-2),
    "Lion":     keras.optimizers.Lion(learning_rate=1e-4, weight_decay=0.1),
}
for nombre, opt in optimizadores.items():
    print(f"{nombre:9s} -> {type(opt).__name__}")

## 2. Hiperparámetros clave: `beta_1`, `beta_2`, `epsilon`, `weight_decay`

Adam usa dos momentos (0.9 / 0.999); Lion no tiene segundo momento (usa el **signo** del gradiente).

In [ ]:
adam = keras.optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
print("Adam  beta_1:", adam.beta_1, "beta_2:", adam.beta_2, "epsilon:", adam.epsilon)

lion = keras.optimizers.Lion(learning_rate=1e-4, beta_1=0.9, beta_2=0.99)
print("Lion  beta_1:", lion.beta_1, "beta_2:", lion.beta_2, "(sin segundo momento)")

## 3. Comparar optimizadores entrenando el mismo MLP

In [ ]:
def mlp(optimizer):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(300, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(100, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for nombre in ["SGD", "Momentum", "Adam", "AdamW", "Lion"]:
    print(f"{nombre:9s} compilado (params {mlp(optimizadores[nombre]).count_params()})")
    # mlp(optimizadores[nombre]).fit(X_tr, y_tr, epochs=20, validation_split=0.1)

## 4. AdamW (decoupled) vs Adam + L2 (coupled)

Loshchilov & Hutter (2019) mostraron que aplicar weight decay separado del gradiente (**AdamW**) supera a Adam con regularización L2 en la loss.

In [ ]:
adamw = mlp(keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-2))

con_l2 = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(300, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2)),
    layers.Dense(100, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2)),
    layers.Dense(10,  activation="softmax"),
])
con_l2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("AdamW (decoupled) suele superar a Adam+L2 (coupled) en val_loss")

## 5. Buffers del optimizer: Adam guarda `m` y `v` por peso

Por eso Adam usa ~2× la memoria de los pesos; Lion solo mantiene `m`.

In [ ]:
modelo = mlp(keras.optimizers.Adam(1e-3))
X = tf.constant(np.random.default_rng(0).normal(size=(64, 784)), dtype=tf.float32)
y = tf.constant(np.random.default_rng(0).integers(0, 10, size=64))
modelo.fit(X, y, epochs=1, verbose=0)                # inicializa los slots del optimizer
print("variables del optimizer Adam (incluye m y v por peso):",
      len(modelo.optimizer.variables))

## Ejercicios

1. **Comparar 5 optimizadores**: SGD, SGD+Momentum, Adam, AdamW, Lion sobre el mismo modelo; graficá `val_loss`.
2. **Tuning del LR**: sweep log de LR para Adam y Lion; verificá que el óptimo de Lion es ~5× más chico.
3. **AdamW vs Adam+L2**: compará `val_loss`.
4. **Inspección de buffers**: imprimí `optimizer.variables` para Adam (m, v) y Lion (solo m).

## Conclusiones

- **Momentum/Nesterov** aceleran en direcciones consistentes; **AdaGrad/RMSProp** adaptan el LR por parámetro.
- **Adam** = momentum + RMSProp + bias correction: el caballito industrial.
- **AdamW** desacopla el weight decay del gradiente: preferilo siempre que uses weight decay.
- **Lion** (2023) usa el signo del gradiente, menos memoria y LR 3-10× más bajo que Adam.
- **SGD+Momentum+cosine** puede generalizar mejor en visión clásica con datasets grandes.